Lab 4: LLMs and Prompt Engineering for Decision Support

Duration: 2 weeks [30 Jul - 13 Aug, 2026] Due Date: 13th August, 2026 Format: Jupyter Notebook / Google Colab + external APIs + GitHub version control Grading: This is a graded lab.

Student Name: Nii Sowah Student ID: 18682028

Objective

In the previous labs you trained models. In this lab you will use a model that someone else spent millions of dollars training — a Large Language Model (LLM) — and learn that getting good results out of one is an engineering discipline of its own: prompt engineering.

You will build a decision support system for a microfinance loan officer. Given a pile of free-text loan application letters, your system will:

    Summarize each application into a short, factual brief,
    Extract specific structured data points (JSON) that a downstream system could store,
    Produce a decision-support recommendation — while keeping the human firmly in the loop.

Just as importantly, you will evaluate the LLM's output for quality, reliability, and appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to make the final call?

Choosing an API provider

You need an LLM API with a free tier. Recommended options (pick ONE):
Provider 	Free tier 	Notes
Groq (recommended) 	Yes, generous 	OpenAI-compatible API, very fast, open models (Llama)
Google Gemini 	Yes 	google-generativeai package
Hugging Face Inference API 	Yes, limited 	Many open models
OpenAI / Anthropic 	Paid 	Fine if you already have credits

The notebook's example code uses the OpenAI-compatible chat format (works with Groq and OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is provider-agnostic.

Part 0: Repository and API-key setup

    Create a public repository named lab-4-llm-decision-support and save this notebook inside it.
    Sign up with your chosen provider and create an API key.
    NEVER hard-code or commit your API key. This is a graded requirement.
        Locally: put it in a .env file and add .env to .gitignore.
        Colab: use the Secrets panel (key icon) and read it with google.colab.userdata.
    Add a requirements.txt: openai python-dotenv pandas matplotlib.
    Commit and push after each Part — we will check for incremental commits.

    A leaked key in your commit history = resubmission + penalty. Keys can be scraped from public repos within minutes.


In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")
     

Client ready.


Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: messages and roles (system, user, assistant), and the generation parameters (temperature, max_tokens).
Part 1.1 — Your first API call

In [4]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
print(ask_llm("What is the capital of France?"))
# TODO: Print response.usage as well — how many tokens did your call consume?


The capital of France is Paris.


Student Reasoning — Anatomy of a call 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

    Answer: A system role is used to set the behavior of the assistant, while a user role is used to provide input or ask questions. For example, a system message could be "You are a helpful assistant that provides concise answers," while a user message could be "What is the capital of France?" A token is roughly a word or a piece of a word, and API providers bill per token because it reflects the amount of computation and resources used to generate the response.


Part 1.2 — Temperature: the randomness dial

In [5]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
     
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0 ")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"\n Run {i+1} ")
    print(answer)

print("\n\n Temperature = 1.2")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"\n Run {i+1} ")
    print(answer)

Temperature = 0.0 

 Run 1 
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good fortune" or "prosperity" in the Akan language, which could be an attractive name for a savings product.
7. **Accra Trader's Fund**: This name is straightforward and emphasizes the product'

Student Reasoning — Temperature What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

    Answer: At temperature 0, the model produces deterministic and consistent responses, which is suitable for decision-support systems where reliability is crucial. At temperature 1.2, the model produces more diverse and creative responses, but with less consistency. Therefore, a lower temperature is appropriate for the loan decision-support system to ensure accurate and reliable recommendations.


Section 2 — The Dataset: Loan Application Letters

Run the next cell to load six loan application letters submitted to a (fictional) microfinance institution in Ghana, plus gold-standard extraction labels for three of them (you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you have not read.

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go. Keep every major prompt version — Section 3.4 asks you to commit your prompt templates and document how they evolved.

Part 3.1 — Component 1: Summarization

Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [8]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this: {letter_text}"

def summarize_v1(letter_text):
    return ask_llm(SUMMARY_PROMPT_V1.format(letter_text=letter_text))

print(" L002 (V1) ")
print(summarize_v1(LETTERS["L002"]))

print("\n L006 (V1) ")
print(summarize_v1(LETTERS["L006"]))

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_SYSTEM = """You are an assistant to a microfinance loan officer. Summarize the loan application factually and neutrally in 3-4 sentences, without inventing any details. You should focus on the applicant's name, requested loan amount, purpose of the loan, monthly profit (if mentioned), whether they have collateral or a guarantor, and proposed repayment terms. """

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

def summarize_v2(letter_text):
    return ask_llm(SUMMARY_PROMPT_V2.format(letter_text=letter_text), system_prompt=SUMMARY_PROMPT_SYSTEM, temperature=0.0)

print(" L002 (V2) ")
print(summarize_v2(LETTERS["L002"]))

print("\n L006 (V2) ")
print(summarize_v2(LETTERS["L006"]))

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

 L002 (V1) 
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can, despite not having collateral at the moment.

 L006 (V1) 
Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan in one year, once his businesses are successful, but has no collateral to offer, relying on his personal guarantee of being trustworthy.
 L002 (V2) 
Kwame Boateng has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. As a commercial driver, he expects his business to improve after the festive season. He does not have collateral to

Student Reasoning — Summarization prompts 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

    Answer: V1's output had issues with hallucination and lack of clarity. For example, it might have added details about the applicant's financial situation that were not present in the original letter. V2 fixed this by providing clearer instructions to summarize only the information present in the letter. "No invented details" is essential to ensure that the summary is accurate and reliable, as adding false information could lead to incorrect loan decisions. This failure mode is known as "hallucination" in LLM literature.


Part 3.2 — Component 2: Structured extraction (JSON)

Downstream software cannot read prose. Extract the fields in GOLD as strict JSON.